# Context Engineering와 장기 메모리
- 컨텍스트가 길어질 때 토큰 예산을 관리하고, 세션 밖 사용자 정보를 `Store`에 저장한 뒤 필요할 때 회상한다.

## 전체 흐름

| 단계 | 핵심 |
|---|---|
| Token count | 메시지가 실제로 얼마나 비싼지 측정 |
| Trim | 토큰 예산에 맞게 메시지 목록 축소 |
| Store | 세션 밖 장기 메모리 저장 |
| Recall middleware | 저장 정보를 LLM 호출 직전 주입 |

이 파일에서는 먼저 대화 컨텍스트를 줄이는 방법을 보고, 이어서 잘라내면 안 되는 사용자 정보를 장기 메모리로 옮기는 방법을 다룹니다.


## 환경 준비

상위 폴더와 동일한 `.env`를 공유합니다. LLM 호출 예제는 `OPENAI_API_KEY`가 필요합니다.

```text
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=mtvs2026-context-eng
```


In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 1. Context Engineering


### 1.1. 토큰이 왜 문제인가, 직접 세보기

- `tiktoken`으로 메시지 토큰 수를 추정합니다. 
- 대화, 문서, 도구 결과가 길어질수록 호출 비용과 지연 시간이 늘어나므로 먼저 측정 가능한 형태로 만들어야 합니다.


In [2]:
import tiktoken
from langchain_core.messages import BaseMessage

enc = tiktoken.encoding_for_model("gpt-4o")     # 토크나이저는 같은 계열이라 공유

def message_text(message: BaseMessage) -> str:
    """메시지 본문을 토큰 계산용 문자열로 변환합니다."""
    content = message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            block.get("text", str(block)) if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content)

def count_tokens(text: str) -> int:
    """텍스트 토큰 수를 추정합니다.(셀 안 디버깅용)"""
    return len(enc.encode(text))

def count_message_tokens(messages: list[BaseMessage]) -> int:
    """메시지 리스트의 총 토큰 수. trim_messages 의 token_counter 인자에 넘김."""
    return sum(count_tokens(message_text(message)) for message in messages)


sample_ko = "안녕하세요, 저는 오늘 에이전트를 배웠어요."
sample_en = "Hello, I am studying about Agent today "

print(f"한국어 문장 토큰: {count_tokens(sample_ko)}")
print(f"영어 문장 토큰: {count_tokens(sample_en)}")


한국어 문장 토큰: 13
영어 문장 토큰: 9


> 한국어는 영어보다 토큰이 보통 1.5~2배 듭니다. 한국어 챗봇이 영어 대비 비용이 더 나가는 이유.

### 1.2. Trim, 오래된 메시지 잘라내기

- `trim_messages`는 메시지 목록을 토큰 예산 안에 맞춥니다. 
- `strategy="last"`는 최근 메시지를 남기므로 챗봇 기본 전략으로 적합합니다.


In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, trim_messages

messages = [
    SystemMessage("너는 사용자의 일정과 취향을 기억해서 짧게 답하는 개인 비서야."),
    HumanMessage("안녕. 나는 다음 주에 제주도로 3박 4일 여행을 가."),
    AIMessage("좋아요. 제주 3박 4일 일정 계획을 도와드릴게요."),
    HumanMessage("첫째 날은 공항 근처에서 렌터카를 받고 동쪽으로 이동할 예정이야."),
    AIMessage("첫째 날은 함덕이나 월정리처럼 이동 부담이 적은 동쪽 코스가 좋습니다."),
    HumanMessage("나는 해산물을 좋아하지만 매운 음식은 잘 못 먹어."),
    AIMessage("해산물 위주로 추천하되 매운탕이나 강한 양념 메뉴는 피하겠습니다."),
    HumanMessage("둘째 날에는 성산일출봉과 우도를 가고 싶어."),
    AIMessage("둘째 날은 성산일출봉을 먼저 보고, 날씨가 좋으면 우도 왕복 일정을 넣으면 좋습니다."),
    HumanMessage("셋째 날에는 서쪽으로 넘어가서 카페와 노을 명소를 보고 싶어."),
    AIMessage("셋째 날은 애월, 협재, 금능 해변 쪽으로 이동하면 카페와 노을을 함께 보기 좋습니다."),
    HumanMessage("이제 지금까지 말한 조건을 바탕으로 마지막 날 오전 일정을 추천해줘."),
]

def show_messages(title: str, message_list: list[BaseMessage]) -> None:
    print(f"\n[{title}] 메시지 {len(message_list)}개, 토큰 {count_message_tokens(message_list)}개")
    for idx, message in enumerate(message_list, start=1):
        print(
            f"{idx:02d}. {type(message).__name__:<13} "
            f"{count_tokens(message_text(message)):>3} tokens | {message.content[:55]}"
        )

show_messages("원본", messages)

# 원본 전체 토큰보다 작게 잡아야 실제로 trim이 일어남
token_budget = 115
trimmed_last = trim_messages(
    messages,
    max_tokens=token_budget,
    token_counter=count_message_tokens,
    strategy='last',                # 최근 메시지 우선 보존
    include_system=True,            # system 메시지는 유지
    allow_partial=False             # 메시지를 중간에서 자르지 않고 통째로 보존/삭제
)


show_messages(f"trim 후, strategy='last', max_tokens={token_budget}", trimmed_last)

kept_texts = {message_text(message) for message in trimmed_last}
dropped_messages = [message for message in messages if message_text(message) not in kept_texts]
show_messages("잘려 나간 메시지", dropped_messages)



[원본] 메시지 12개, 토큰 266개
01. SystemMessage  21 tokens | 너는 사용자의 일정과 취향을 기억해서 짧게 답하는 개인 비서야.
02. HumanMessage   19 tokens | 안녕. 나는 다음 주에 제주도로 3박 4일 여행을 가.
03. AIMessage      20 tokens | 좋아요. 제주 3박 4일 일정 계획을 도와드릴게요.
04. HumanMessage   24 tokens | 첫째 날은 공항 근처에서 렌터카를 받고 동쪽으로 이동할 예정이야.
05. AIMessage      24 tokens | 첫째 날은 함덕이나 월정리처럼 이동 부담이 적은 동쪽 코스가 좋습니다.
06. HumanMessage   15 tokens | 나는 해산물을 좋아하지만 매운 음식은 잘 못 먹어.
07. AIMessage      23 tokens | 해산물 위주로 추천하되 매운탕이나 강한 양념 메뉴는 피하겠습니다.
08. HumanMessage   17 tokens | 둘째 날에는 성산일출봉과 우도를 가고 싶어.
09. AIMessage      29 tokens | 둘째 날은 성산일출봉을 먼저 보고, 날씨가 좋으면 우도 왕복 일정을 넣으면 좋습니다.
10. HumanMessage   23 tokens | 셋째 날에는 서쪽으로 넘어가서 카페와 노을 명소를 보고 싶어.
11. AIMessage      31 tokens | 셋째 날은 애월, 협재, 금능 해변 쪽으로 이동하면 카페와 노을을 함께 보기 좋습니다.
12. HumanMessage   20 tokens | 이제 지금까지 말한 조건을 바탕으로 마지막 날 오전 일정을 추천해줘.

[trim 후, strategy='last', max_tokens=115] 메시지 4개, 토큰 95개
01. SystemMessage  21 tokens | 너는 사용자의 일정과 취향을 기억해서 짧게 답하는 개인 비서야.
02. HumanMessage   23 tokens | 셋째 날에는 서쪽으로 넘어

### 1.3. 다른 전략, `strategy="first"`

- 처음 입력된 요구사항이나 원본 문서가 더 중요한 작업에서는 앞쪽 메시지를 남기는 `first` 전략이 유용합니다.

    - `last`: 최근 맥락이 중요한 챗봇에 적합
    - `first`: 초반 요구사항이나 원본 지시가 더 중요한 작업에 적합

주의: `include_system=True`는 `strategy="last"`와 함께 쓸 때의 옵션입니다. `first`에서는 시스템 메시지가 앞에 있으면 자연스럽게 먼저 보존됩니다.


In [4]:
trimmed_first = trim_messages(
    messages,
    max_tokens=token_budget,
    token_counter=count_message_tokens,
    strategy='first',                # 초반 메시지 유지
    allow_partial=False              # 메시지를 중간에서 자르지 않고 통째로 보존/삭제
)

show_messages(f"trim 후, strategy='first', max_tokens={token_budget}", trimmed_first)


[trim 후, strategy='first', max_tokens=115] 메시지 5개, 토큰 108개
01. SystemMessage  21 tokens | 너는 사용자의 일정과 취향을 기억해서 짧게 답하는 개인 비서야.
02. HumanMessage   19 tokens | 안녕. 나는 다음 주에 제주도로 3박 4일 여행을 가.
03. AIMessage      20 tokens | 좋아요. 제주 3박 4일 일정 계획을 도와드릴게요.
04. HumanMessage   24 tokens | 첫째 날은 공항 근처에서 렌터카를 받고 동쪽으로 이동할 예정이야.
05. AIMessage      24 tokens | 첫째 날은 함덕이나 월정리처럼 이동 부담이 적은 동쪽 코스가 좋습니다.


### 1.4. 컨텍스트 설계 체크리스트

| 질문 | 권장 답 |
|---|---|
| 시스템 프롬프트가 너무 길지 않나? | 핵심 규칙만 남기고 도구 설명은 도구 docstring으로 이동 |
| 매 호출마다 같은 정보를 다시 넣고 있지 않나? | 캐시 또는 메모리에서 필요할 때만 가져오기 |
| 대화가 50턴 이상 누적되는가? | Trim + Summarization 조합 사용 |
| RAG 결과를 통째로 넣는가? | 압축, 재정렬, 상위 청크만 선택 |
| 사용자가 한 번 알려준 정보를 다시 묻는가? | 장기 메모리에 저장 후 회상 |


## 2. Long-term Memory


- 지금까지 본 `checkpointer` 는 **한 대화 세션 안** 의 상태만 보존했습니다. 사용자가 어제 알려준 "내 캐릭터 이름은 도윤" 같은 정보를 **다음 날 새 세션** 에서도 기억하게 하려면 **장기 메모리 (Store)** 가 필요합니다.
- LangGraph 의 `InMemoryStore` 가 가장 간단한 시작점. 운영에서는 PostgreSQL / Redis 백엔드로 교체.

### 2.1. Store의 3가지 키

- `Store`는 세션 밖에 정보를 저장합니다. 
- `namespace`, `key`, `value`를 분리하면 사용자별, 카테고리별 데이터를 깔끔하게 관리할 수 있습니다.


In [5]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# put(namespace, key, value)
# namespace 는 튜플, 사용자별 / 카테고리별 폴더 같은 개념
store.put(
    ("user-doyun", "profile"),
    "character",
    {"name": "도윤", "class": "마법사", "level": 12},
)

store.put(
    ("user-doyun", "preference"),
    "ui",
    {"theme": "dark", "language": "ko"},
)

store.put(
    ("user-seoa", "profile"),
    "character",
    {"name": "서아", "class": "기사", "level": 28},
)

# 조회
item = store.get(("user-doyun", "profile"), "character")
print("도윤 캐릭터:", item.value)

item2 = store.get(("user-seoa", "profile"), "character")
print("서아 캐릭터:", item2.value)

도윤 캐릭터: {'name': '도윤', 'class': '마법사', 'level': 12}
서아 캐릭터: {'name': '서아', 'class': '기사', 'level': 28}


### 2.2. Namespace 단위로 검색


In [6]:
# 도윤 의 모든 데이터
items = store.search(("user-doyun",))
print(f"도윤 관련 데이터 {len(items)} 개")
for item in items:
    print(f"  namespace={item.namespace}, key={item.key}, value={item.value}")

도윤 관련 데이터 2 개
  namespace=('user-doyun', 'profile'), key=character, value={'name': '도윤', 'class': '마법사', 'level': 12}
  namespace=('user-doyun', 'preference'), key=ui, value={'theme': 'dark', 'language': 'ko'}


### 2.3. 에이전트에 Store 연결

- `create_agent(store=store)`로 전달하면 미들웨어에서 `runtime.store`로 장기 메모리에 접근할 수 있습니다. 아래 미들웨어는 LLM 호출 직전에 저장된 사용자 정보를 시스템 메시지로 주입합니다.


In [7]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langgraph.runtime import Runtime

llm = init_chat_model("openai:gpt-4.1-mini")

class MemoryRecallMiddleware(AgentMiddleware):
    """매 LLM 호출 직전, 저장된 사용자 프로필을 시스템 메시지로 주입합니다."""

    # 미들웨어를 만들 때 어떤 사용자의 정보를 읽을지 user_id를 받음
    def __init__(self, user_id: str):
        super().__init__()
        self.user_id = user_id

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if runtime.store is None:    
            return None
        items = runtime.store.search((self.user_id,)) # 현재 사용자 ID에 해당하는 모든 장기 메모리 검색
        if not items:       # 저장된 정보가 없으면 아무것도 하지 않음
            return None
        
        # 검색된 메모리들을 LLM이 읽기 쉬운 문자열로 변환
        summary = "\n".join(f"- {item.namespace}/{item.key}: {item.value}" for item in items)
        
        # 저장 정보를 시스템 메시지로 만듦. 이 메시지가 LLM 입력 앞쪽에 추가됨.
        recall = SystemMessage(f"사용자 저장 정보:\n{summary}")

        # LLM은 사용자 질문을 보기 전에 저장된 사용자 정보를 먼저 보게됨
        return {"messages": [recall] + list(state["messages"])}


@tool
def remember(key: str, value: str) -> str:
    """사용자 정보를 장기 메모리에 저장합니다. 예: key='favorite_color', value='파랑'."""
    return f"기억하겠습니다: {key} = {value}"


memory_agent = create_agent(
    model=llm,
    tools=[remember],
    system_prompt="너는 사용자를 기억하는 챗봇입니다.",
    middleware=[MemoryRecallMiddleware(user_id="user-doyun")], # 모델 호출 직전에 store에서 사용자 정보를 꺼내오는 middleware 등록
    store=store
)


### 2.4. 새 세션에서도 정보 회상


In [ ]:
# 실행 방식이 invoke/stream 중 어떤 방식이든 동작하도록 처리
try:
    result = memory_agent.invoke(
        {"messages": [HumanMessage("내 캐릭터 정보 알려줘")]}
    )
    print(result["messages"][-1].content)
except Exception as e:
    print(f"invoke 방식 실패: {e}")
    try:
        result = memory_agent.stream(
            {"messages": [HumanMessage("내 캐릭터 정보 알려줘")]},
            stream_mode="values",
        )
        for step in result:
            if "messages" in step:
                print(step["messages"][-1].content)
    except Exception as e2:
        print(f"stream 방식도 실패: {e2}")

당신의 캐릭터 정보는 다음과 같습니다.
이름: 도윤
직업: 마법사
레벨: 12

더 궁금한 점이 있으면 알려주세요!


### 2.5. 세션 상태와 장기 메모리 비교

| 개념 | 역할 | 예시 |
|---|---|---|
| Checkpointer | 한 대화 thread 안의 중간 상태 저장 | interrupt 후 같은 `thread_id`로 재개 |
| Store | 세션 밖 사용자 정보 저장 | 다음 날 새 thread에서도 취향, 프로필 회상 |
| Recall middleware | 저장된 정보를 LLM 입력에 주입 | `runtime.store.search(...)` 결과를 `SystemMessage`로 추가 |
| 운영용 Store | 프로세스 재시작 후에도 유지 | PostgreSQL, Redis 등으로 교체 |


## 정리와 실습

### 정리

- 컨텍스트는 한정 자원입니다. 긴 대화, RAG 결과, 도구 결과를 모두 넣으면 비용과 지연 시간이 커집니다.
- `count_message_tokens`처럼 먼저 측정 가능한 토큰 카운터를 만들어야 trim 기준을 잡을 수 있습니다.
- `trim_messages(strategy="last")`는 최근 대화를 유지하는 챗봇 기본 전략입니다.
- `strategy="first"`는 초반 지시나 원본 문서가 더 중요한 작업에 적합합니다.
- 짧게 잘라내면 안 되는 사용자 정보는 `Store`에 저장하고, 필요할 때 `runtime.store`로 회상합니다.
- `checkpointer`는 세션 안 상태, `Store`는 세션 밖 장기 메모리를 담당합니다.

### 실습

1. `token_budget`을 40, 200으로 바꿔 `last`와 `first` 결과를 비교하세요.
2. `include_system=False`로 바꿔 시스템 메시지 보존 여부를 확인하세요.
3. `user-minjun` 데이터를 추가하고 `MemoryRecallMiddleware(user_id="user-minjun")`으로 응답을 비교하세요.
4. `store.search(("user-doyun",))` 결과를 요약하는 규칙을 바꿔, LLM에 들어가는 recall 메시지를 더 짧게 만들어보세요.
